# 02 Travel-Time Fit and Calibrated Parameters

Rebuilds the best-fit model from the archived calibration output and produces
the travel-time and parameter results.

## What This Notebook Produces

Measured against predicted first-arrival times for all seven profiles, with the RMSE distribution across the behavioral ensemble 

Behavioral distributions of the seven calibrated parameters against their search bounds, with the median and best fit marked 


Run `01_calibration_and_validation.ipynb` first, or use the
`outputs/dc_sceua_pc055_lb` directory supplied with this repository. Output is
written to `_figures_local/`, which is excluded from version control.


## 1. Imports, Paths, Optional Backends, and Style

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
import numpy as np
import pandas as pd
from scipy.interpolate import RegularGridInterpolator

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().resolve().parents[0]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_io import read_config
from src.dem_tools import sample_dem_along_line
from src.sceua_landlab_rd_rpv import (
    SCEUA_PARAMETER_NAMES,
    build_line_section_from_model,
    build_model_from_candidate,
    get_line_split,
    load_sceua_base_inputs,
    load_sceua_config,
    normalize_sceua_parameter_frame,
)

ENV_PREFIX = Path(sys.executable).resolve().parent
for extra_path in (ENV_PREFIX / "Library" / "bin", ENV_PREFIX / "Scripts", ENV_PREFIX / "bin"):
    if extra_path.exists():
        os.environ["PATH"] = str(extra_path) + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("PYGMT_USE_EXTERNAL_DISPLAY", "false")

try:
    import pyvista as pv
    HAS_PYVISTA = True
except Exception as exc:
    pv = None
    HAS_PYVISTA = False
    PYVISTA_IMPORT_ERROR = exc

try:
    import pygmt
    HAS_PYGMT = True
except Exception as exc:
    pygmt = None
    HAS_PYGMT = False
    PYGMT_IMPORT_ERROR = exc

try:
    import cmcrameri.cm as cmc
    DEPTH_CMAP = cmc.batlow
except Exception:
    DEPTH_CMAP = "viridis"

PROJECT_CONFIG = read_config(ROOT / "config.yaml")
RESULT_DIR = ROOT / PROJECT_CONFIG["sceua_landlab_rd_rpv"]["output_dir"]
DATA_DIR = RESULT_DIR / "data"
FIG_DIR = ROOT / PROJECT_CONFIG["paper_inputs"]["figure_output_dir"]
SAVE_FIGURES = True
FIG_DPI = 600

TRAINING_LINES = [
    "TL1",
    "TL2",
    "TL3",
    "TL4",
    "TL5",
]
VALIDATION_LINES = ["TL6", "TL7"]
ALL_LINES = TRAINING_LINES + VALIDATION_LINES
LINE_LABELS = {
    "TL1": "TL-1",
    "TL2": "TL-2",
    "TL3": "TL-3",
    "TL4": "TL-4",
    "TL5": "TL-5",
    "TL6": "TL-6",
    "TL7": "TL-7",
}
PARAMETER_LABELS = {
    "P0": r"$P_0$",
    "Hs": r"$H_s$",
    "D": r"$D$",
    "r": r"$r$",
    "phi_soil_top": r"$\phi_{m,top}$",
    "phi_weathered_top": r"$\phi_{w,top}$",
    "phi_fresh": r"$\phi_f$",
}
DATASET_COLORS = {
    "calibration": "#1f78b4",
    "validation": "#d95f02",
}

def windows_safe_path(path: str | Path) -> str:
    """Return a path string that can read or write long paths on Windows."""
    path = Path(path)
    if os.name != "nt":
        return str(path)
    text = str(path.resolve())
    if len(text) < 240 or text.startswith("\\\\?\\"):
        return text
    if text.startswith("\\\\"):
        return "\\\\?\\UNC\\" + text[2:]
    return "\\\\?\\" + text


def output_exists(path: str | Path) -> bool:
    return os.path.exists(windows_safe_path(path))


def read_output_csv(path: str | Path, **kwargs) -> pd.DataFrame:
    return pd.read_csv(windows_safe_path(path), **kwargs)



def normalize_parameter_summary_table(frame: pd.DataFrame) -> pd.DataFrame:
    """Convert old Kd/gamma row labels to current D/Hs labels for plotting."""
    out = frame.copy()
    if "parameter" not in out.columns:
        return out
    parameters = set(out["parameter"].astype(str))
    if "D" not in parameters and "Kd" in parameters:
        out.loc[out["parameter"] == "Kd", "parameter"] = "D"
    if "Hs" not in parameters and "gamma" in parameters:
        mask = out["parameter"] == "gamma"
        out.loc[mask, "parameter"] = "Hs"
        for low_col, high_col in (("lower", "upper"), ("p05", "p95")):
            if low_col in out.columns and high_col in out.columns:
                old_low = out.loc[mask, low_col].astype(float).copy()
                old_high = out.loc[mask, high_col].astype(float).copy()
                out.loc[mask, low_col] = 1.0 / old_high
                out.loc[mask, high_col] = 1.0 / old_low
        for col in ("median", "best_fit"):
            if col in out.columns:
                out.loc[mask, col] = 1.0 / out.loc[mask, col].astype(float)
        if "mean" in out.columns:
            out.loc[mask, "mean"] = np.nan
    return out

def load_output_npz(path: str | Path) -> dict[str, np.ndarray]:
    with np.load(windows_safe_path(path)) as npz:
        return {key: npz[key] for key in npz.files}


required = [
    DATA_DIR / "best_fit_parameters.csv",
    DATA_DIR / "behavioral_parameter_sets.csv",
    DATA_DIR / "behavioral_3d_uncertainty.npz",
    DATA_DIR / "paper_parameter_summary.csv",
    DATA_DIR / "sceua_results.csv",
    DATA_DIR / "training_behavioral_prediction_summary.csv",
    DATA_DIR / "validation_behavioral_prediction_summary.csv",
]
required.extend(
    DATA_DIR / f"training_behavioral_{line_id}_prediction.npz"
    for line_id in TRAINING_LINES
)
required.extend(
    DATA_DIR / f"validation_behavioral_{line_id}_prediction.npz"
    for line_id in VALIDATION_LINES
)
missing = [str(p) for p in required if not output_exists(p)]
if missing:
    raise FileNotFoundError("Missing required notebook 11 outputs:\n" + "\n".join(missing))

if SAVE_FIGURES:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": FIG_DPI,
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

VP_CMAP = "magma"
VP_DISPLAY_MIN_M_PER_S = 350.0
VP_DISPLAY_BREAK_M_PER_S = 700.0
VP_DISPLAY_MAX_M_PER_S = 4000.0
VP_LOW_FRACTION = 0.48
VP_COLORBAR_TICKS = [350, 450, 550, 700, 1200, 2000, 3200, 3500, 4000]


def _low_vp_forward(values):
    arr = np.asarray(values, dtype=float)
    out = np.empty_like(arr, dtype=float)
    low = arr <= VP_DISPLAY_BREAK_M_PER_S
    out[low] = VP_LOW_FRACTION * (
        (arr[low] - VP_DISPLAY_MIN_M_PER_S)
        / (VP_DISPLAY_BREAK_M_PER_S - VP_DISPLAY_MIN_M_PER_S)
    )
    out[~low] = VP_LOW_FRACTION + (1.0 - VP_LOW_FRACTION) * (
        (arr[~low] - VP_DISPLAY_BREAK_M_PER_S)
        / (VP_DISPLAY_MAX_M_PER_S - VP_DISPLAY_BREAK_M_PER_S)
    )
    return out


def _low_vp_inverse(values):
    arr = np.asarray(values, dtype=float)
    out = np.empty_like(arr, dtype=float)
    low = arr <= VP_LOW_FRACTION
    out[low] = VP_DISPLAY_MIN_M_PER_S + (arr[low] / VP_LOW_FRACTION) * (
        VP_DISPLAY_BREAK_M_PER_S - VP_DISPLAY_MIN_M_PER_S
    )
    out[~low] = VP_DISPLAY_BREAK_M_PER_S + (
        (arr[~low] - VP_LOW_FRACTION) / (1.0 - VP_LOW_FRACTION)
    ) * (VP_DISPLAY_MAX_M_PER_S - VP_DISPLAY_BREAK_M_PER_S)
    return out


VP_NORM = mcolors.FuncNorm(
    (_low_vp_forward, _low_vp_inverse),
    vmin=VP_DISPLAY_MIN_M_PER_S,
    vmax=VP_DISPLAY_MAX_M_PER_S,
)
SECTION_DEPTH_MAX_M = 24.0
SECTION_MIN_DISPLAY_DEPTH_M = 8.0
SECTION_FRESHBEDROCK_MARGIN_M = 1.5
SECTION_FRESH_INTERFACE_PERCENTILE = 88.0
SECTION_TOP_PADDING_M = 1.5
TRAVELTIME_BINS_S = np.array([0.00, 0.02, 0.04, 0.06, 0.08, 0.10, 0.13], dtype=float)


def panel_label(ax, label: str, x: float = 0.015, y: float = 0.965, color: str = "black") -> None:
    return None


def save_candidate(fig: mpl.figure.Figure, name: str) -> None:
    if not SAVE_FIGURES:
        return
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(windows_safe_path(FIG_DIR / f"{name}.png"), dpi=FIG_DPI, bbox_inches="tight")
    fig.savefig(windows_safe_path(FIG_DIR / f"{name}.pdf"), bbox_inches="tight")


def strip_axes(ax) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

print(f"Optional backends: PyVista={HAS_PYVISTA}, PyGMT={HAS_PYGMT}. SAVE_FIGURES={SAVE_FIGURES}")

## 2. Load Results and Rebuild the Best-Fit Model

In [ ]:
def load_line_fit_summary() -> pd.DataFrame:
    """Load the freshest line-fit table from behavioral prediction summaries."""
    summary_specs = [
        (DATA_DIR / "training_behavioral_prediction_summary.csv", "calibration"),
        (DATA_DIR / "validation_behavioral_prediction_summary.csv", "validation"),
    ]
    if all(output_exists(path) for path, _ in summary_specs):
        frames = []
        for path, dataset in summary_specs:
            frame = read_output_csv(path).copy()
            frame["dataset"] = dataset
            frames.append(frame)
        combined = pd.concat(frames, ignore_index=True)
    else:
        fallback = DATA_DIR / "paper_line_fit_summary.csv"
        if not output_exists(fallback):
            missing = [str(path) for path, _ in summary_specs if not output_exists(path)]
            raise FileNotFoundError(
                "Missing prediction summary outputs and fallback paper_line_fit_summary.csv:\n"
                + "\n".join(missing)
            )
        combined = read_output_csv(fallback).copy()
        if "dataset" not in combined.columns:
            combined["dataset"] = np.where(
                combined["line"].isin(TRAINING_LINES), "calibration", "validation"
            )
    order = {line_id: index for index, line_id in enumerate(ALL_LINES)}
    combined["line_label"] = combined["line"].map(LINE_LABELS)
    combined["line_order"] = combined["line"].map(order)
    combined = combined.sort_values("line_order").drop(columns="line_order")
    return combined.reset_index(drop=True)


best_parameters = normalize_sceua_parameter_frame(read_output_csv(DATA_DIR / "best_fit_parameters.csv")).iloc[0]
behavioral_sets = normalize_sceua_parameter_frame(read_output_csv(DATA_DIR / "behavioral_parameter_sets.csv"))
line_fit = load_line_fit_summary()
parameter_summary = normalize_parameter_summary_table(read_output_csv(DATA_DIR / "paper_parameter_summary.csv"))
sceua_results = normalize_sceua_parameter_frame(read_output_csv(DATA_DIR / "sceua_results.csv"))
uncertainty_3d = load_output_npz(DATA_DIR / "behavioral_3d_uncertainty.npz")
best_candidate = {name: float(best_parameters[name]) for name in SCEUA_PARAMETER_NAMES}
config = read_config(ROOT / "config.yaml")
sceua_config = load_sceua_config(config)
training_lines, validation_lines = get_line_split(config)
assert training_lines == TRAINING_LINES
assert validation_lines == VALIDATION_LINES
assert sceua_config["output_dir"] == str(RESULT_DIR.relative_to(ROOT)).replace("\\", "/")

base_inputs = load_sceua_base_inputs(config, ROOT, line_names=ALL_LINES)
assert base_inputs["rpv_params"]["soil"]["critical_porosity"] == 0.55
assert base_inputs["rpv_params"]["soil"]["hertz_mindlin_bound"] == "lower"
assert base_inputs["rpv_params"]["weathered_bedrock"]["alpha_top"] == 0.015
assert base_inputs["rpv_params"]["weathered_bedrock"]["alpha_bottom"] == 0.015
assert base_inputs["rpv_params"]["fresh_bedrock"]["alpha"] == 0.015
best_model = build_model_from_candidate(best_candidate, base_inputs, config)

dem = base_inputs["dem"]
x = np.asarray(dem.x, dtype=float)
y = np.asarray(dem.y, dtype=float)
X, Y = np.meshgrid(x - x.min(), y - y.min())
surface_abs = np.asarray(dem.data, dtype=float)
h_soil_mean = np.asarray(uncertainty_3d["H_soil_mean"], dtype=float)
d_fresh_p50 = np.asarray(uncertainty_3d["D_fresh_p50"], dtype=float)
fresh_depth_norm = mcolors.Normalize(
    vmin=float(np.nanmin(d_fresh_p50)),
    vmax=float(np.nanmax(d_fresh_p50)),
)
soil_base_abs = surface_abs - h_soil_mean
fresh_interface_abs = surface_abs - d_fresh_p50

print("Best-fit parameters")
display(pd.DataFrame([best_candidate]))
print("Line fit summary")
line_fit_display_cols = [
    col for col in ["line", "line_label", "dataset", "rmse_s", "coverage_90", "n_picks"]
    if col in line_fit.columns
]
display(line_fit[line_fit_display_cols])
print("Model grid", best_model["vp"].shape, "depth max", float(np.max(best_model["z"])))

## 3. Load Travel-Time Prediction Outputs

In [ ]:
def prediction_prefix(line_id: str) -> str:
    return "training_behavioral" if line_id in TRAINING_LINES else "validation_behavioral"


def prediction_dataset(line_id: str) -> str:
    return "calibration" if line_id in TRAINING_LINES else "validation"


def load_prediction_output(line_id: str) -> dict[str, np.ndarray | str]:
    prefix = prediction_prefix(line_id)
    path = DATA_DIR / f"{prefix}_{line_id}_prediction.npz"
    if not output_exists(path):
        raise FileNotFoundError(path)
    data = load_output_npz(path)
    data["line"] = line_id
    data["line_label"] = LINE_LABELS[line_id]
    data["dataset"] = prediction_dataset(line_id)
    return data


def sample_summary_candidates(line_id: str) -> list[Path]:
    prefix = prediction_prefix(line_id)
    candidates = [DATA_DIR / f"{prefix}_{line_id}_sample_summary.csv"]
    if line_id == "TL6":
        candidates.append(DATA_DIR / "validation_behavioral_TL6_sample_summary.csv")
    return candidates


def load_prediction_sample_summary(line_id: str) -> pd.DataFrame:
    for path in sample_summary_candidates(line_id):
        if output_exists(path):
            frame = read_output_csv(path).copy()
            frame["line"] = line_id
            frame["line_label"] = LINE_LABELS[line_id]
            frame["dataset"] = prediction_dataset(line_id)
            frame["source_file"] = path.name
            return frame
    raise FileNotFoundError(
        "No sample summary found for "
        + line_id
        + ": "
        + ", ".join(str(path) for path in sample_summary_candidates(line_id))
    )


prediction_data = {line_id: load_prediction_output(line_id) for line_id in ALL_LINES}
prediction_sample_summaries = {
    line_id: load_prediction_sample_summary(line_id) for line_id in ALL_LINES
}
print("Loaded prediction files:", ", ".join(prediction_data))
print(
    "Loaded sample summaries:",
    ", ".join(
        f"{LINE_LABELS[line_id]}={len(frame)} sets"
        for line_id, frame in prediction_sample_summaries.items()
    ),
)


## 4. Extract Terrain-Following Vp Sections and Behavioral Interface Envelopes

In [ ]:
N_BEHAVIORAL_FOR_INTERFACES = None  # None uses all retained behavioral sets.


def dem_surface_for_line(line_id: str) -> np.ndarray:
    line_xy = np.asarray(base_inputs["lines"][line_id]["line_xy"], dtype=float)
    surface = sample_dem_along_line(line_xy[:, 0], line_xy[:, 1], dem)
    if not np.all(np.isfinite(surface)):
        n_bad = int(np.sum(~np.isfinite(surface)))
        raise ValueError(f"DEM sampling failed for {line_id}: {n_bad} non-finite elevation values.")
    return surface


def restore_section_surface_from_dem(line_id: str, section: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    out = dict(section)
    out["surface_elevation"] = dem_surface_for_line(line_id)
    return out



def extract_sections_for_model(model: dict) -> dict[str, dict[str, np.ndarray]]:
    return {
        line_id: restore_section_surface_from_dem(
            line_id,
            build_line_section_from_model(model, base_inputs["lines"][line_id], base_inputs),
        )
        for line_id in ALL_LINES
    }

best_sections = extract_sections_for_model(best_model)

interface_samples: dict[str, dict[str, list[np.ndarray]]] = {
    line_id: {"H_soil": [], "D_fresh": []} for line_id in ALL_LINES
}
behavioral_interface_sets = (
    behavioral_sets
    if N_BEHAVIORAL_FOR_INTERFACES is None
    else behavioral_sets.head(N_BEHAVIORAL_FOR_INTERFACES)
)
for _, row in behavioral_interface_sets.iterrows():
    sample_model = build_model_from_candidate(row, base_inputs, config)
    for line_id in ALL_LINES:
        section = build_line_section_from_model(sample_model, base_inputs["lines"][line_id], base_inputs)
        interface_samples[line_id]["H_soil"].append(np.asarray(section["H_soil"], dtype=float))
        interface_samples[line_id]["D_fresh"].append(np.asarray(section["D_fresh"], dtype=float))

interface_envelopes = {}
for line_id, groups in interface_samples.items():
    interface_envelopes[line_id] = {}
    for key, arrays in groups.items():
        stack = np.stack(arrays)
        interface_envelopes[line_id][key] = {
            "p05": np.nanpercentile(stack, 5, axis=0),
            "p50": np.nanpercentile(stack, 50, axis=0),
            "p95": np.nanpercentile(stack, 95, axis=0),
        }

surface_check_rows = []
for line_id in ALL_LINES:
    dem_surface = np.asarray(best_sections[line_id]["surface_elevation"], dtype=float)
    sensors = base_inputs["lines"][line_id]["tt_data"]["sensors"].sort_values("x")
    relative_surface = sensors["elevation"].to_numpy(dtype=float)
    surface_check_rows.append({
        "line": line_id,
        "DEM_elevation_min_m": float(np.nanmin(dem_surface)),
        "DEM_elevation_max_m": float(np.nanmax(dem_surface)),
        "seismic_relative_min_m": float(np.nanmin(relative_surface)),
        "seismic_relative_max_m": float(np.nanmax(relative_surface)),
    })
surface_elevation_check = pd.DataFrame(surface_check_rows)

print(f"Extracted {len(best_sections)} best-fit line sections and {len(behavioral_interface_sets)} behavioral interface samples.")
print("Section surface elevations restored from topo-derived x/y positions sampled on the DEM.")
display(surface_elevation_check)

## 5. Plot Helpers

In [ ]:
def line_dataset(line_id: str) -> str:
    return "calibration" if line_id in TRAINING_LINES else "validation"


def line_metric(line_id: str, column: str = "rmse_s") -> float:
    metric = line_fit.loc[line_fit["line"].eq(line_id), column]
    return float(metric.iloc[0]) if len(metric) else np.nan


def line_rmse_s(line_id: str) -> float:
    return line_metric(line_id, "rmse_s")


def section_display_depth(line_id: str, section: dict[str, np.ndarray]) -> float:
    """Choose a shallow-focused display depth for each line section."""
    d_best = np.asarray(section["D_fresh"], dtype=float)
    target = max(
        SECTION_MIN_DISPLAY_DEPTH_M,
        float(np.nanpercentile(d_best, SECTION_FRESH_INTERFACE_PERCENTILE))
        + SECTION_FRESHBEDROCK_MARGIN_M,
    )
    return float(min(SECTION_DEPTH_MAX_M, target))


def plot_vp_section(
    ax,
    line_id: str,
    section: dict[str, np.ndarray],
    *,
    show_ylabel: bool = True,
    show_xlabel: bool = True,
    title_suffix: str = "",
) -> mpl.collections.QuadMesh:
    distance = np.asarray(section["distance"], dtype=float)
    depth = np.asarray(section["depth"], dtype=float)
    vp = np.asarray(section["vp"], dtype=float)
    display_depth = section_display_depth(line_id, section)
    keep_depth = depth <= display_depth
    depth_plot = depth[keep_depth]
    vp_plot = vp[keep_depth]
    surface = np.asarray(section["surface_elevation"], dtype=float)
    elevation_grid = surface[None, :] - depth_plot[:, None]
    distance_grid = np.tile(distance[None, :], (len(depth_plot), 1))

    im = ax.pcolormesh(
        distance_grid,
        elevation_grid,
        vp_plot,
        shading="auto",
        cmap=VP_CMAP,
        norm=VP_NORM,
        rasterized=True,
    )
    env = interface_envelopes[line_id]
    h_soil = np.asarray(section["H_soil"], dtype=float)
    d_fresh = np.asarray(section["D_fresh"], dtype=float)

    ax.fill_between(
        distance,
        surface - env["D_fresh"]["p95"],
        surface - env["D_fresh"]["p05"],
        color="#d9d9d9",
        alpha=0.48,
        lw=0,
        label="$D_f$ p05-p95",
        zorder=3,
    )
    ax.plot(distance, surface, color="black", lw=1.0, label="topography", zorder=5)
    ax.plot(
        distance,
        surface - h_soil,
        color="white",
        lw=1.05,
        alpha=0.98,
        label="base of mobile regolith",
        zorder=6,
    )
    ax.plot(
        distance,
        surface - d_fresh,
        color="#00bcd4",
        lw=1.15,
        alpha=0.98,
        label="fresh interface",
        zorder=6,
    )

    zmin = float(np.nanmin(surface - display_depth))
    zmax = float(np.nanmax(surface + SECTION_TOP_PADDING_M))
    ax.set_ylim(zmin, zmax)
    ax.set_xlim(float(np.nanmin(distance)), float(np.nanmax(distance)))
    if show_ylabel:
        ax.set_ylabel("Elevation (m)")
    else:
        ax.set_ylabel("")
        ax.tick_params(labelleft=False)
    if show_xlabel:
        ax.set_xlabel("Distance along line (m)")
    else:
        ax.set_xlabel("")
        ax.tick_params(labelbottom=False)
    group = line_dataset(line_id)
    ax.set_title(f"{LINE_LABELS[line_id]} ({group}, RMSE={line_rmse_s(line_id):.4f} s){title_suffix}", pad=2)
    strip_axes(ax)
    return im


def plot_traveltime_fit(ax, line_id: str) -> None:
    """Measured versus predicted travel-time plot, avoiding offset as the x axis."""
    data = prediction_data[line_id]
    observed_s = np.asarray(data["observed"], dtype=float)
    p05_s = np.asarray(data["p05"], dtype=float)
    p50_s = np.asarray(data["p50"], dtype=float)
    p95_s = np.asarray(data["p95"], dtype=float)
    valid = np.isfinite(observed_s) & np.isfinite(p50_s)
    observed_s = observed_s[valid]
    p05_s = p05_s[valid]
    p50_s = p50_s[valid]
    p95_s = p95_s[valid]
    color = DATASET_COLORS[line_dataset(line_id)]

    order = np.argsort(observed_s)
    step = max(1, len(order) // 180)
    thin = order[::step]
    ax.vlines(
        observed_s[thin],
        p05_s[thin],
        p95_s[thin],
        color=color,
        alpha=0.16,
        lw=0.45,
        rasterized=True,
        label="behavioral p05-p95",
    )
    ax.scatter(
        observed_s,
        p50_s,
        s=5.2,
        color=color,
        alpha=0.36,
        linewidths=0,
        rasterized=True,
        label="behavioral median",
    )
    lim_min = 0.0
    lim_max = float(np.nanmax(np.r_[observed_s, p95_s])) * 1.04
    ax.plot([lim_min, lim_max], [lim_min, lim_max], color="0.15", lw=0.75, ls="--", label="1:1")
    ax.set_xlim(lim_min, lim_max)
    ax.set_ylim(lim_min, lim_max)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Measured travel time (s)")
    ax.set_ylabel("Predicted travel time (s)")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(3))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(3))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    strip_axes(ax)


def binned_residual_summary(line_id: str) -> pd.DataFrame:
    data = prediction_data[line_id]
    measured_s = np.asarray(data["observed"], dtype=float)
    residual_s = np.asarray(data["p50"], dtype=float) - np.asarray(data["observed"], dtype=float)
    rows = []
    for lo, hi in zip(TRAVELTIME_BINS_S[:-1], TRAVELTIME_BINS_S[1:]):
        mask = (measured_s >= lo) & (measured_s < hi)
        if not np.any(mask):
            continue
        rows.append({
            "line": line_id,
            "dataset": line_dataset(line_id),
            "time_mid_s": 0.5 * (lo + hi),
            "time_label_s": f"{lo:.2f}-{hi:.2f}",
            "n": int(mask.sum()),
            "p10": float(np.nanpercentile(residual_s[mask], 10)),
            "p50": float(np.nanpercentile(residual_s[mask], 50)),
            "p90": float(np.nanpercentile(residual_s[mask], 90)),
            "rmse": float(np.sqrt(np.nanmean(residual_s[mask] ** 2))),
        })
    return pd.DataFrame(rows)


def plot_residual_bins(ax, line_id: str) -> None:
    table = binned_residual_summary(line_id)
    ax.axhline(0.0, color="0.2", lw=0.7)
    color = DATASET_COLORS[line_dataset(line_id)]
    ax.vlines(table["time_mid_s"], table["p10"], table["p90"], color="0.58", lw=1.0)
    ax.scatter(table["time_mid_s"], table["p50"], s=18, color=color, zorder=3)
    ax.set_title(LINE_LABELS[line_id], pad=2)
    ax.set_xlabel("Measured travel time (s)")
    ax.set_ylabel("Pred. - obs. (s)")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(3))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(3))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    strip_axes(ax)


def plot_map(
    ax,
    values: np.ndarray,
    title: str,
    *,
    cmap=DEPTH_CMAP,
    vmin: float | None = None,
    vmax: float | None = None,
    add_lines: bool = True,
):
    xx = x - x.min()
    yy = y - y.min()
    im = ax.pcolormesh(xx, yy, values, shading="auto", cmap=cmap, vmin=vmin, vmax=vmax, rasterized=True)
    if add_lines:
        for line_id in ALL_LINES:
            line_xy = np.asarray(base_inputs["lines"][line_id]["line_xy"], dtype=float)
            ax.plot(line_xy[:, 0] - x.min(), line_xy[:, 1] - y.min(), lw=0.8, color="white", alpha=0.9)
            mid = line_xy[len(line_xy) // 2]
            ax.text(mid[0] - x.min(), mid[1] - y.min(), LINE_LABELS[line_id], fontsize=5.5, color="white")
    ax.set_aspect("equal")
    ax.set_title(title, pad=3)
    ax.set_xlabel("Easting offset (m)")
    ax.set_ylabel("Northing offset (m)")
    strip_axes(ax)
    return im


def add_shared_colorbar(fig, mappable, axes, label: str, *, orientation: str = "vertical", pad: float = 0.02):
    cbar = fig.colorbar(mappable, ax=axes, orientation=orientation, pad=pad, fraction=0.035)
    cbar.set_label(label)
    if label.startswith("Vp"):
        cbar.set_ticks(VP_COLORBAR_TICKS)
        cbar.set_ticklabels([str(tick) for tick in VP_COLORBAR_TICKS])
        cbar.ax.tick_params(labelsize=7, width=0.5, length=2.0)
    cbar.outline.set_linewidth(0.5)
    return cbar


## Travel-Time Fit and RMSE Across the Behavioral Ensemble

In [ ]:
fig = plt.figure(figsize=(13.5, 5.8), constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.06, wspace=0.04, hspace=0.14)

gs = fig.add_gridspec(2, 7)
tt_axes   = [fig.add_subplot(gs[0, j]) for j in range(7)]
hist_axes = [fig.add_subplot(gs[1, j]) for j in range(7)]

# ── Row 0: travel-time scatter ────────────────────────────────────────────────
for j, line_id in enumerate(ALL_LINES):
    ax = tt_axes[j]
    plot_traveltime_fit(ax, line_id)
    ax.set_title("")
    if j != 0:
        ax.set_ylabel("")
        ax.tick_params(axis="y", labelleft=False)
    else:
        ax.set_ylabel("Predicted travel time (s)", fontsize=10)
    ax.set_xlabel("Measured travel time (s)", fontsize=10)
    ax.tick_params(axis="x", labelbottom=True, labelsize=9)
    ax.tick_params(axis="y", labelsize=9)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(3))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(3))
    ax.text(0.5, 1.02, LINE_LABELS[line_id], transform=ax.transAxes,
            ha="center", va="bottom", fontsize=10,
            color=DATASET_COLORS[line_dataset(line_id)])

# ── Row 1: RMSE histograms ────────────────────────────────────────────────────
for j, line_id in enumerate(ALL_LINES):
    ax = hist_axes[j]
    sample_rmse = prediction_sample_summaries[line_id]["rmse_s"].to_numpy(dtype=float)
    best_rmse   = line_metric(line_id, "rmse_s")
    median_rmse = float(np.nanmedian(sample_rmse))
    color       = DATASET_COLORS[line_dataset(line_id)]

    nbins = max(10, len(sample_rmse) // 6)
    ax.hist(sample_rmse, bins=nbins, color=color, alpha=0.72,
            edgecolor="white", linewidth=0.5, density=False)
    ax.axvline(best_rmse,   color="red", lw=1.3, ls=":",  zorder=4)
    ax.axvline(median_rmse, color="#07110ecc", lw=1.1, ls=":",  zorder=3)

    ax.set_xlabel("RMSE (s)", labelpad=2, fontsize=10)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(3))
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    ax.tick_params(axis="x", labelsize=9)
    ax.tick_params(axis="y", labelsize=9)
    if j == 0:
        ax.set_ylabel("Count", fontsize=10)
    else:
        ax.set_ylabel("")
        ax.tick_params(axis="y", labelleft=False)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(3, integer=True))
    strip_axes(ax)

# ── Shared legend below ───────────────────────────────────────────────────────
from matplotlib.patches import Patch
legend_handles = [
    mpl.lines.Line2D([0], [0], marker="o", color="w",
                     markerfacecolor=DATASET_COLORS["calibration"], markersize=7,
                     label="Calibration"),
    mpl.lines.Line2D([0], [0], marker="o", color="w",
                     markerfacecolor=DATASET_COLORS["validation"], markersize=7,
                     label="Validation"),
    mpl.lines.Line2D([0], [0], color="0.15", lw=0.9, ls="--", label="1:1 line"),
    mpl.lines.Line2D([0], [0], color="red", lw=1.3, ls=":",  label="Best fit RMSE"),
    mpl.lines.Line2D([0], [0], color="#07110ecc", lw=1.1, ls=":",  label="Median RMSE"),
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.07),
    frameon=False,
    ncol=5,
    fontsize=10,
    handlelength=2.0,
    columnspacing=1.5,
    handletextpad=0.5,
)

save_candidate(fig, "candidate_G_G2_combined_traveltime_rmse")
plt.savefig(windows_safe_path(FIG_DIR / 'traveltime_rmse_combined.tif'), dpi=300, bbox_inches="tight")
plt.show()


## Calibrated Parameter Distributions

In [ ]:
from matplotlib.patches import Patch

PARAMETER_UNITS = {
    "P0":                "(m yr$^{-1}$)",
    "Hs":                "(m)",
    "D":                 "(m$^2$ yr$^{-1}$)",
    "r":                 "(-)",
    "phi_soil_top":      "(-)",
    "phi_weathered_top": "(-)",
    "phi_fresh":         "(-)",
}

plot_params = ["P0", "Hs", "D", "r", "phi_soil_top", "phi_weathered_top", "phi_fresh"]
summary = parameter_summary.set_index("parameter").loc[plot_params].reset_index()
summary["behavioral_width"] = summary["p95"] - summary["p05"]

n_params = len(summary)
fig, axes = plt.subplots(n_params, 1, figsize=(4.5, 7.0), sharex=False, constrained_layout=True)

for i, (ax, row) in enumerate(zip(axes, summary.itertuples(index=False))):
    name = row.parameter
    values = behavioral_sets[name].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    lower = float(row.lower)
    upper = float(row.upper)

    bins = np.linspace(lower, upper, 22)
    ax.hist(values, bins=bins, color="#9ecae1", alpha=0.80,
            edgecolor="white", linewidth=0.5, density=False)

    ax.axvline(lower, color="0.65", lw=0.7, ls=":", zorder=2)
    ax.axvline(upper, color="0.65", lw=0.7, ls=":", zorder=2)
    ax.axvline(float(row.median), color="#08519c", lw=1.4, ls="-", zorder=4)
    ax.axvline(float(row.best_fit), color="#d95f02", lw=1.4, ls="--", zorder=5)

    ax.set_ylabel("Count", labelpad=3)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(3, integer=True))

    unit = PARAMETER_UNITS.get(name, "")
    sym = PARAMETER_LABELS.get(name, name)
    ax.set_xlabel(f"{sym}  {unit}", labelpad=2)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(4))
    if name in {"P0", "D"}:
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2e"))
    else:
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))

    pad = 0.03 * (upper - lower)
    ax.set_xlim(lower - pad, upper + pad)
    strip_axes(ax)

legend_handles = [
    mpl.lines.Line2D([0], [0], color="#08519c", lw=1.4, ls="-", label="Median"),
    mpl.lines.Line2D([0], [0], color="#d95f02", lw=1.4, ls="--", label="Best fit"),
    mpl.lines.Line2D([0], [0], color="0.65", lw=0.7, ls=":", label="Prior bounds"),
]
axes[-1].legend(
    handles=legend_handles,
    frameon=False,
    loc="center",
    fontsize=8,
    handlelength=1.8,
    labelspacing=0.45,
    handletextpad=0.5,
)

parameter_uncertainty_table = summary[[
    "parameter", "lower", "upper", "p05", "median", "p95", "best_fit", "behavioral_width", "near_bound_5pct"
]].copy()
save_candidate(fig, "candidate_I_calibrated_parameter_uncertainty")
plt.savefig(windows_safe_path(FIG_DIR / 'parameter_uncertainty_histograms.tif'), dpi=300, bbox_inches="tight")
plt.show()
display(parameter_uncertainty_table)
